# Pré-processamento

## 1. Imports

In [1]:
import copy
from matplotlib.path import Path as _MplPath
def _fixed_path_deepcopy(self, memo):
    cls = self.__class__
    p = cls.__new__(cls)
    memo[id(self)] = p
    for k, v in self.__dict__.items():
        try:
            setattr(p, k, copy.deepcopy(v, memo))
        except Exception:
            setattr(p, k, v)
    p._readonly = False
    return p
_MplPath.__deepcopy__ = _fixed_path_deepcopy

import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
import joblib
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('models', exist_ok=True)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 2. Carregamento e limpeza inicial

In [2]:
df = pd.read_csv('data/data.csv')
df.drop(columns=['Unnamed: 32', 'id'], inplace=True)

print('Shape:', df.shape)
print('Missing values:', df.isnull().sum().sum())
print('Duplicatas:', df.duplicated().sum())
print('\nClasses:\n', df['diagnosis'].value_counts())

Shape: (569, 31)
Missing values: 0
Duplicatas: 0

Classes:
 diagnosis
B    357
M    212
Name: count, dtype: int64


## 3. Encode do target

In [3]:
le = LabelEncoder()
df['diagnosis'] = le.fit_transform(df['diagnosis'])  # B=0, M=1

print('Mapeamento:', dict(zip(le.classes_, le.transform(le.classes_))))
print('Distribuição após encode:\n', df['diagnosis'].value_counts())

Mapeamento: {'B': np.int64(0), 'M': np.int64(1)}
Distribuição após encode:
 diagnosis
0    357
1    212
Name: count, dtype: int64


/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_28970/3137586183.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['diagnosis'] = le.fit_transform(df['diagnosis'])  # B=0, M=1


M → 1, B → 0. Maligno como classe positiva — assim as métricas de recall/F1 calculam em cima de M por padrão.

## 4. Separação X / y

In [4]:
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

print('Features:', X.shape[1])
print('Amostras:', X.shape[0])

Features: 30
Amostras: 569


## 5. Divisão treino / validação / teste

Três splits pra não abrir o teste durante a escolha de modelos. O teste fica fechado até o final.

In [5]:
# Primeiro: separa 20% para teste final
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Do restante (80%): separa 15% para validação (~12% do total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.15, random_state=42, stratify=y_temp
)

print(f'Treino:    {X_train.shape[0]} amostras ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Validação: {X_val.shape[0]} amostras ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Teste:     {X_test.shape[0]} amostras ({X_test.shape[0]/len(X)*100:.0f}%)')
print()
print('Proporção de malignos (M=1):')
print(f'  Treino:    {y_train.mean():.2%}')
print(f'  Validação: {y_val.mean():.2%}')
print(f'  Teste:     {y_test.mean():.2%}')

Treino:    386 amostras (68%)
Validação: 69 amostras (12%)
Teste:     114 amostras (20%)

Proporção de malignos (M=1):
  Treino:    37.31%
  Validação: 37.68%
  Teste:     36.84%


## 6. Normalização

StandardScaler (média 0, desvio 1). Fit só no treino, transform em todos os conjuntos.

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Converte de volta pra DataFrame pra facilitar inspeção
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print('Média pós-escala (treino):')
print(X_train_scaled.mean().round(6).to_string())
print('\nDesvio padrão pós-escala (treino):')
print(X_train_scaled.std().round(6).to_string())

Média pós-escala (treino):
radius_mean               -0.0
texture_mean              -0.0
perimeter_mean             0.0
area_mean                  0.0
smoothness_mean           -0.0
compactness_mean          -0.0
concavity_mean             0.0
concave points_mean       -0.0
symmetry_mean             -0.0
fractal_dimension_mean    -0.0
radius_se                  0.0
texture_se                 0.0
perimeter_se              -0.0
area_se                   -0.0
smoothness_se              0.0
compactness_se             0.0
concavity_se              -0.0
concave points_se          0.0
symmetry_se               -0.0
fractal_dimension_se       0.0
radius_worst              -0.0
texture_worst              0.0
perimeter_worst           -0.0
area_worst                 0.0
smoothness_worst           0.0
compactness_worst          0.0
concavity_worst           -0.0
concave points_worst      -0.0
symmetry_worst            -0.0
fractal_dimension_worst   -0.0

Desvio padrão pós-escala (treino):
radius_

## 7. Verificação visual — antes vs. depois da normalização

In [7]:
features_spot = ['radius_mean', 'area_mean', 'smoothness_mean', 'fractal_dimension_mean']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, feat in enumerate(features_spot):
    # Antes
    axes[0, i].hist(X_train[feat], bins=30, color='#7fcdbb', edgecolor='white')
    axes[0, i].set_title(f'{feat}\n(original)')
    # Depois
    axes[1, i].hist(X_train_scaled[feat], bins=30, color='#2c7fb8', edgecolor='white')
    axes[1, i].set_title(f'{feat}\n(normalizado)')

axes[0, 0].set_ylabel('Original')
axes[1, 0].set_ylabel('Normalizado')
plt.suptitle('Distribuição antes e depois do StandardScaler', fontsize=13, fontweight='bold')
plt.savefig('figures/06_scaling_comparison.png', bbox_inches='tight')
plt.show()
print('[OK] Figura salva.')

[OK] Figura salva.


/var/folders/c0/pbls_tj56jn3kq8p6gckzcc40000gn/T/ipykernel_28970/1126923077.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Pipeline de pré-processamento

Empacota o scaler num Pipeline do sklearn e salva em disco — facilita aplicar o mesmo processamento em dados novos.

In [8]:
preprocessing_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

preprocessing_pipeline.fit(X_train)

# Salva pra usar nas fases seguintes
joblib.dump(preprocessing_pipeline, 'models/preprocessing_pipeline.pkl')
print('Pipeline salvo em models/preprocessing_pipeline.pkl')
print('\nSteps:', preprocessing_pipeline.steps)

Pipeline salvo em models/preprocessing_pipeline.pkl

Steps: [('scaler', StandardScaler())]


## 9. Salva os splits

Salva os três conjuntos já normalizados pra garantir que as próximas fases usam exatamente os mesmos dados.

In [9]:
os.makedirs('data/processed', exist_ok=True)

X_train_scaled.assign(diagnosis=y_train.values).to_csv('data/processed/train.csv', index=False)
X_val_scaled.assign(diagnosis=y_val.values).to_csv('data/processed/val.csv', index=False)
X_test_scaled.assign(diagnosis=y_test.values).to_csv('data/processed/test.csv', index=False)

print('Splits salvos em data/processed/')
print(f'  train.csv : {X_train_scaled.shape[0]} linhas')
print(f'  val.csv   : {X_val_scaled.shape[0]} linhas')
print(f'  test.csv  : {X_test_scaled.shape[0]} linhas')

Splits salvos em data/processed/
  train.csv : 386 linhas
  val.csv   : 69 linhas
  test.csv  : 114 linhas


## 10. Checklist de saída

In [10]:
checks = {
    'Sem missing no treino': X_train_scaled.isnull().sum().sum() == 0,
    'Sem missing no val':    X_val_scaled.isnull().sum().sum() == 0,
    'Sem missing no teste':  X_test_scaled.isnull().sum().sum() == 0,
    'Média treino ≈ 0':      abs(X_train_scaled.mean().mean()) < 1e-6,
    'Std treino ≈ 1':        abs(X_train_scaled.std(ddof=0).mean() - 1) < 1e-3,
    'Target é 0/1':          set(y_train.unique()).issubset({0, 1}),
    'Splits somam 100%':     (len(X_train) + len(X_val) + len(X_test)) == len(X),
    'Pipeline salvo':        os.path.exists('models/preprocessing_pipeline.pkl'),
}

all_ok = True
for check, result in checks.items():
    status = '✓' if result else '✗'
    print(f'  {status}  {check}')
    if not result:
        all_ok = False

print()
print('Tudo certo. Pronto para modelagem.' if all_ok else 'ATENÇÃO: algum check falhou.')

  ✓  Sem missing no treino
  ✓  Sem missing no val
  ✓  Sem missing no teste
  ✓  Média treino ≈ 0
  ✓  Std treino ≈ 1
  ✓  Target é 0/1
  ✓  Splits somam 100%
  ✓  Pipeline salvo

Tudo certo. Pronto para modelagem.
